# Step 3 - Training an Encoder-Decoder Model (U-Net)
## Semantic Segmentation with Deep Learning - Potsdam Dataset

**Input:** RGB + IR + Elevation (5 bands)  
**Model:** U-Net Encoder-Decoder architecture  
**Training:** 20 epochs, Categorical Cross-Entropy, Best validation model saved  
**Output:** Semantic segmentation prediction maps

## 3.1 Install and Import Libraries

In [ ]:
!pip install rasterio tensorflow matplotlib scikit-learn -q

In [ ]:
import os
import json
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import rasterio
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# =====================================================================
# Hyperparameters
# =====================================================================
SEED            = 42
BATCH_SIZE      = 8          # smaller batch due to larger model
EPOCHS          = 20
LEARNING_RATE   = 1e-4
NUM_CLASSES     = 6
INPUT_CHANNELS  = 5          # RGB + IR + Elevation
MODEL_SAVE_PATH = 'best_unet_model.h5'

CLASS_NAMES = [
    'Impervious surface', 'Building', 'Tree',
    'Low vegetation', 'Car', 'Clutter/Background'
]
CLASS_COLORS = [
    [255, 255, 255], [0, 0, 255], [0, 255, 0],
    [0, 255, 255],   [255, 255, 0], [255, 0, 0]
]

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('Libraries imported!')
print(f'TensorFlow: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

## 3.2 Load Fold Splits

In [ ]:
import os, random, json
import numpy as np
import matplotlib
matplotlib.use('Agg')  # headless mode
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import rasterio
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import ModelCheckpoint
from sklearn.model_selection import KFold
from IPython.display import Image, display

# ── Reproducibility seed ────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Global settings ────────────────────────
N_FOLDS     = 5
NUM_CLASSES = 6

def discover_data_dir():
    possible = [
        os.path.join(os.getcwd(), 'Potsdam-GeoTif'),
        os.path.join(os.getcwd(), 'data'),
        os.path.join(os.getcwd(), 'PROJECT', 'Potsdam-GeoTif'),
        os.path.join(os.getcwd(), 'PROJECT', 'data'),
        os.getcwd()
    ]
    for p in [p for p in possible if os.path.exists(p)]:
        try:
            if any(f.endswith('.tif') for f in os.listdir(p)):
                return p
        except:
            continue
    return 'data'

DATA_DIR = discover_data_dir()
print(f"Data directory: {DATA_DIR}")


## 3.3 Data Preprocessing and Augmentation

In [ ]:
def read_geotiff(file_path):
    with rasterio.open(file_path) as src:
        return src.read()  # (bands, H, W)

def normalize_band(band):
    b_min, b_max = band.min(), band.max()
    if b_max == b_min:
        return np.zeros_like(band, dtype=np.float32)
    return (band - b_min) / (b_max - b_min)

def load_sample(file_path, use_all_bands=True):
    data = read_geotiff(file_path)
    n_bands = 5 if use_all_bands else 4
    features = data[:n_bands].transpose(1, 2, 0).astype(np.float32)
    for c in range(features.shape[-1]):
        features[..., c] = normalize_band(features[..., c])
    label_band = data[5].astype(np.int32)
    label_onehot = tf.keras.utils.to_categorical(label_band, num_classes=NUM_CLASSES)
    return features, label_onehot

def augment(image, label):
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_left_right(image)
        label = tf.image.flip_left_right(label)
    if tf.random.uniform(()) > 0.5:
        image = tf.image.flip_up_down(image)
        label = tf.image.flip_up_down(label)
    k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)
    image = tf.image.rot90(image, k)
    label = tf.image.rot90(label, k)
    return image, label

def make_dataset(file_paths, use_all_bands=True, augment_data=False, batch_size=8, shuffle=True):
    def _load(fp):
        fp_str = fp.numpy().decode('utf-8')
        return load_sample(fp_str, use_all_bands=use_all_bands)
    def _tf_load(fp):
        features, labels = tf.py_function(_load, [fp], [tf.float32, tf.float32])
        return features, labels
    ds = tf.data.Dataset.from_tensor_slices(file_paths)
    if shuffle:
        ds = ds.shuffle(buffer_size=len(file_paths), seed=SEED)
    ds = ds.map(_tf_load, num_parallel_calls=tf.data.AUTOTUNE)
    if augment_data:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_files, use_all_bands=True, augment_data=True,  batch_size=BATCH_SIZE, shuffle=True)
val_ds   = make_dataset(val_files,   use_all_bands=True, augment_data=False, batch_size=BATCH_SIZE, shuffle=False)
test_ds  = make_dataset(test_files,  use_all_bands=True, augment_data=False, batch_size=BATCH_SIZE, shuffle=False)

print('Datasets ready!')

## 3.4 U-Net Encoder-Decoder Architecture

In [ ]:
def conv_block(x, filters, name_prefix):
    """Two Conv2D + BN + ReLU layers."""
    x = layers.Conv2D(filters, 3, padding='same', activation='relu', name=f'{name_prefix}_conv1')(x)
    x = layers.BatchNormalization(name=f'{name_prefix}_bn1')(x)
    x = layers.Conv2D(filters, 3, padding='same', activation='relu', name=f'{name_prefix}_conv2')(x)
    x = layers.BatchNormalization(name=f'{name_prefix}_bn2')(x)
    return x


def build_unet(input_channels=5, num_classes=6, base_filters=32):
    """
    U-Net Encoder-Decoder for semantic segmentation.
    
    Architecture:
    Encoder: 4 blocks with MaxPooling (downsampling)
    Bottleneck
    Decoder: 4 blocks with UpSampling + skip connections
    Output: 1x1 Conv + Softmax
    """
    inputs = keras.Input(shape=(None, None, input_channels), name='input')
    
    # ===--- ENCODER ---===
    e1 = conv_block(inputs, base_filters * 1, 'enc1')          # 32 filters
    p1 = layers.MaxPooling2D(2, name='pool1')(e1)
    
    e2 = conv_block(p1, base_filters * 2, 'enc2')              # 64 filters
    p2 = layers.MaxPooling2D(2, name='pool2')(e2)
    
    e3 = conv_block(p2, base_filters * 4, 'enc3')              # 128 filters
    p3 = layers.MaxPooling2D(2, name='pool3')(e3)
    
    e4 = conv_block(p3, base_filters * 8, 'enc4')              # 256 filters
    p4 = layers.MaxPooling2D(2, name='pool4')(e4)
    
    # ===--- BOTTLENECK ---===
    b = conv_block(p4, base_filters * 16, 'bottleneck')        # 512 filters
    
    # ===--- DECODER ---===
    u4 = layers.UpSampling2D(2, name='up4')(b)
    u4 = layers.Concatenate(name='skip4')([u4, e4])
    d4 = conv_block(u4, base_filters * 8, 'dec4')
    
    u3 = layers.UpSampling2D(2, name='up3')(d4)
    u3 = layers.Concatenate(name='skip3')([u3, e3])
    d3 = conv_block(u3, base_filters * 4, 'dec3')
    
    u2 = layers.UpSampling2D(2, name='up2')(d3)
    u2 = layers.Concatenate(name='skip2')([u2, e2])
    d2 = conv_block(u2, base_filters * 2, 'dec2')
    
    u1 = layers.UpSampling2D(2, name='up1')(d2)
    u1 = layers.Concatenate(name='skip1')([u1, e1])
    d1 = conv_block(u1, base_filters * 1, 'dec1')
    
    # ===--- OUTPUT ---===
    outputs = layers.Conv2D(num_classes, 1, padding='same', activation='softmax', name='output')(d1)
    
    model = keras.Model(inputs, outputs, name='UNet_SegModel')
    return model


unet_model = build_unet(input_channels=INPUT_CHANNELS, num_classes=NUM_CLASSES)
unet_model.summary()

## 3.5 Visualize U-Net Architecture

In [ ]:
try:
    import visualkeras
    img = visualkeras.layered_view(unet_model, legend=True, to_file='unet_architecture.png')
    plt.figure(figsize=(14, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title('U-Net Encoder-Decoder Architecture')
    plt.show()
except Exception:
    keras.utils.plot_model(unet_model, to_file='unet_architecture.png',
                           show_shapes=True, show_layer_names=True, dpi=60)
    from IPython.display import Image, display
    display(Image('unet_architecture.png'))

print('U-Net architecture saved to unet_architecture.png')

## 3.6 Compile and Train the U-Net Model

In [ ]:
unet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    ModelCheckpoint(
        filepath=MODEL_SAVE_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    )
]

print(f'U-Net compiled. Training for {EPOCHS} epochs...')
print(f'Best model will be saved to: {MODEL_SAVE_PATH}')

In [ ]:
# Train
history = unet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

## 3.7 Plot Training History

In [ ]:
def plot_history(history, title_prefix=''):
    epochs_range = range(1, len(history.history['loss']) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(epochs_range, history.history['loss'],     label='Training Loss',   color='blue')
    ax1.plot(epochs_range, history.history['val_loss'], label='Validation Loss', color='orange')
    ax1.set_title(f'{title_prefix} Loss Curve')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2.plot(epochs_range, history.history['accuracy'],     label='Training Accuracy',   color='green')
    ax2.plot(epochs_range, history.history['val_accuracy'], label='Validation Accuracy', color='red')
    ax2.set_title(f'{title_prefix} Accuracy Curve')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
    ax2.legend(); ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{title_prefix.replace(" ", "_")}_training_curves.png', dpi=150)
    plt.show()

plot_history(history, title_prefix='U-Net')

## 3.8 Evaluate on Test Set

In [ ]:
best_model = keras.models.load_model(MODEL_SAVE_PATH)

print('Evaluating on test set...')
test_loss, test_accuracy = best_model.evaluate(test_ds, verbose=1)

print(f'\n===== U-Net Test Set Results =====')
print(f'Test Loss    : {test_loss:.4f}')
print(f'Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)')

## 3.9 Visualize Predictions

**Assignment Requirement:** Visualize RGB, Elevation, Target Label, and **Prediction Map** side by side.

In [ ]:
def label_to_rgb(label_band, colors):
    """Convert label indices to RGB visualization."""
    h, w = label_band.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for class_idx, color in enumerate(colors):
        rgb[label_band == class_idx] = color
    return rgb

def normalize_band(band):
    b_min, b_max = band.min(), band.max()
    if b_max == b_min:
        return np.zeros_like(band, dtype=np.float32)
    return (band - b_min) / (b_max - b_min)


# Select a test sample for visualization
sample_test_file = test_files[0]
print(f'Visualizing: {os.path.basename(sample_test_file)}')

# Load raw data
with rasterio.open(sample_test_file) as src:
    raw_data = src.read()  # (6, H, W)

# Prepare input (5 bands)
features_raw = raw_data[:5].transpose(1, 2, 0).astype(np.float32)
for c in range(features_raw.shape[-1]):
    features_raw[..., c] = normalize_band(features_raw[..., c])

# GT label
gt_label = raw_data[5].astype(np.int32)

# Model prediction
model_input = np.expand_dims(features_raw, axis=0)   # (1, H, W, 5)
pred_logits = best_model.predict(model_input)[0]      # (H, W, 6)
pred_label  = np.argmax(pred_logits, axis=-1)         # (H, W)

# Visualization
rgb_vis    = np.stack([normalize_band(raw_data[0].astype(np.float32)),
                       normalize_band(raw_data[1].astype(np.float32)),
                       normalize_band(raw_data[2].astype(np.float32))], axis=-1)
elev_vis   = normalize_band(raw_data[4].astype(np.float32))
gt_rgb     = label_to_rgb(gt_label,   CLASS_COLORS)
pred_rgb   = label_to_rgb(pred_label, CLASS_COLORS)

# Legend patches
patches = [mpatches.Patch(color=[c/255 for c in CLASS_COLORS[i]], label=CLASS_NAMES[i])
           for i in range(NUM_CLASSES)]

fig, axes = plt.subplots(1, 4, figsize=(22, 6))
fig.suptitle(f'U-Net Prediction - {os.path.basename(sample_test_file)}', fontsize=13)

axes[0].imshow(rgb_vis)
axes[0].set_title('RGB Image'); axes[0].axis('off')

im_e = axes[1].imshow(elev_vis, cmap='terrain')
axes[1].set_title('Elevation Band'); axes[1].axis('off')
plt.colorbar(im_e, ax=axes[1], fraction=0.046, pad=0.04, label='Normalized Elevation')

axes[2].imshow(gt_rgb)
axes[2].set_title('Ground Truth Label'); axes[2].axis('off')
axes[2].legend(handles=patches, loc='lower right', fontsize=6, framealpha=0.8)

axes[3].imshow(pred_rgb)
axes[3].set_title('U-Net Prediction'); axes[3].axis('off')
axes[3].legend(handles=patches, loc='lower right', fontsize=6, framealpha=0.8)

plt.tight_layout()
plt.savefig('unet_prediction_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('Prediction visualization saved to unet_prediction_visualization.png')

## 3.10 Summary

✅ U-Net Encoder-Decoder model built with 4 encoder/decoder blocks + bottleneck  
✅ Skip connections between encoder and decoder paths  
✅ Trained for 20 epochs with RGB + IR + Elevation (5 bands)  
✅ Best model saved based on validation accuracy  
✅ Training curves plotted (loss & accuracy)  
✅ Model evaluated on test set  
✅ Prediction visualization: RGB, Elevation, Ground Truth Label, Prediction Map  

---

## Assumptions Made
| Hyperparameter | Value | Reason |
|---|---|---|
| Batch size | 8 | Smaller due to larger U-Net memory footprint |
| Learning rate | 1e-4 | Common for U-Net training |
| Base filters | 32 | Balance between capacity and speed |
| Epochs | 20 | As specified in assignment |
| Augmentation | Flip + Rotate | Improves generalization for aerial imagery |
| Loss function | Categorical Cross-Entropy | As specified in assignment |